# 404 — Program Annotation

## Objective

Biologically characterize the three frozen cross-system consensus transcriptomic
representations using prespecified pathway- and process-level annotation
frameworks.

The goal is to determine which biological processes are preferentially
represented across the ranked gene-weight profiles of each consensus program
and to establish an interpretable annotation layer for downstream functional,
pharmacogenomic, and perturbational analyses.

## Analytical status

Notebook 404 operates exclusively on the frozen consensus representations
constructed in notebook 401.

Consensus-program identities, orientations, gene weights, source-system
correspondences, lineage-robustness results, and the negative/limited
epigenetic-regulator enrichment results from notebook 403 are upstream evidence
and are not modified here.

Annotation results cannot be used to redefine, rescue, exclude, reweight, or
rename a consensus program post hoc.

## Annotation framework

Biological annotation will be performed on the complete ranked consensus-gene
profiles rather than on arbitrarily selected top-gene cutoffs.

Gene-set resources and statistical parameters will be frozen before inspecting
enrichment results. Multiple-testing correction will be applied within
prespecified annotation families.

Interpretation will prioritize:

- effect magnitude and rank consistency;
- coherent biological themes rather than isolated significant terms;
- concordance or divergence across the three consensus programs;
- source-system and lineage context where necessary; and
- explicit separation between biological annotation and mechanistic inference.

## Scope and methodological boundary

This notebook does not:

- modify consensus-program construction;
- search additional annotation databases after observing results;
- interpret pathway enrichment as causal pathway activation;
- infer regulator activity or signaling direction without supporting evidence;
- use pharmacogenomic, dependency, or perturbational outcomes;
- promote individual genes or pathways as validated targets; or
- reinterpret annotation significance as biological validation.

The resulting annotations are descriptive and hypothesis-generating. They are
intended to clarify the biological identity of the frozen consensus programs
and provide a stable interpretation layer for subsequent phases.

## Expected output

Notebook 404 will publish one downstream-consumable biological-annotation
artifact under `data/processed/consensus_programs/`.

Intermediate annotation tables, redundant gene-set catalogs, and exploratory
figures will not be persisted unless they become necessary downstream.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.utils.paths import Paths

In [2]:
# =============================================================================
# Input and output directories
# =============================================================================

CONSENSUS_PROGRAM_DIR = Paths.consensus_programs
OUTPUT_DIR = Paths.consensus_programs
MSIGDB_DIR = Paths.msigdb

In [3]:
# =============================================================================
# Authoritative program-annotation input paths
# =============================================================================

CONSENSUS_CATALOG_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_transcriptomic_program_catalog.csv"
)

CONSENSUS_GENE_WEIGHTS_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_transcriptomic_gene_weights.csv"
)

In [4]:
# =============================================================================
# Load authoritative program-annotation inputs
# =============================================================================

consensus_catalog = pd.read_csv(
    CONSENSUS_CATALOG_PATH
)

consensus_gene_weights = pd.read_csv(
    CONSENSUS_GENE_WEIGHTS_PATH
)

In [5]:
# =============================================================================
# Freeze consensus-program annotation set
# =============================================================================

consensus_programs = (
    consensus_catalog[
        [
            "consensus_program_id",
            "tumor_rna_axis",
            "cell_line_program",
            "orientation_multiplier",
        ]
    ]
    .copy()
)

consensus_program_ids = consensus_programs[
    "consensus_program_id"
].tolist()

consensus_programs

,consensus_program_id,tumor_rna_axis,cell_line_program,orientation_multiplier
0,CONSENSUS_TX_01,RNA_IC150,ICA_PROGRAM_09,1
1,CONSENSUS_TX_02,RNA_IC151,ICA_PROGRAM_29,-1
2,CONSENSUS_TX_03,RNA_IC184,ICA_PROGRAM_13,-1


In [6]:
# =============================================================================
# Freeze signed consensus-program ranking
# =============================================================================

ranked_consensus_gene_weights = (
    consensus_gene_weights.loc[
        consensus_gene_weights["consensus_program_id"].isin(
            consensus_program_ids
        ),
        [
            "consensus_program_id",
            "gene_symbol",
            "consensus_weight",
        ],
    ]
    .sort_values(
        ["consensus_program_id", "consensus_weight"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

In [7]:
# =============================================================================
# Freeze consensus-gene annotation universe
# =============================================================================

consensus_gene_universe = (
    ranked_consensus_gene_weights[["gene_symbol"]]
    .drop_duplicates()
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

consensus_gene_universe.shape

(2389, 1)

In [8]:
# =============================================================================
# Freeze biological-annotation design
# =============================================================================

MSIGDB_RELEASE = "v2026.1.Hs"

ANNOTATION_COLLECTIONS = {
    "HALLMARK": {
        "filename": "h.all.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "primary",
    },
    "REACTOME": {
        "filename": "c2.cp.reactome.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "secondary",
    },
    "GO_BP": {
        "filename": "c5.go.bp.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "exploratory",
    },
}

MIN_GENE_SET_SIZE = 10
MAX_GENE_SET_SIZE = 500
FDR_ALPHA = 0.05

In [9]:
# =============================================================================
# MSigDB annotation resource paths
# =============================================================================

annotation_resource_paths = {
    collection_name: MSIGDB_DIR / collection_config["filename"]
    for collection_name, collection_config in ANNOTATION_COLLECTIONS.items()
}

In [10]:
# =============================================================================
# Define GMT parser
# =============================================================================

def read_gmt(path):
    """Read an MSigDB GMT file as a mapping from gene-set name to symbols."""
    gene_sets = {}

    with path.open(encoding="utf-8") as handle:
        for line in handle:
            fields = line.rstrip("\n").split("\t")
            gene_sets[fields[0]] = set(fields[2:])

    return gene_sets

In [11]:
# =============================================================================
# Load MSigDB annotation collections
# =============================================================================

annotation_gene_sets = {
    collection_name: read_gmt(resource_path)
    for collection_name, resource_path in annotation_resource_paths.items()
}

In [12]:
# =============================================================================
# Summarize loaded annotation collections
# =============================================================================

annotation_collection_summary = pd.DataFrame(
    [
        {
            "collection": collection_name,
            "analysis_tier": ANNOTATION_COLLECTIONS[collection_name]["analysis_tier"],
            "gene_set_count": len(gene_sets),
        }
        for collection_name, gene_sets in annotation_gene_sets.items()
    ]
)

annotation_collection_summary

,collection,analysis_tier,gene_set_count
0,HALLMARK,primary,50
1,REACTOME,secondary,1839
2,GO_BP,exploratory,7538


In [13]:
# =============================================================================
# Restrict gene sets to the frozen annotation universe
# =============================================================================

consensus_gene_universe_set = set(consensus_gene_universe["gene_symbol"])

eligible_annotation_gene_sets = {}

for collection_name, gene_sets in annotation_gene_sets.items():
    eligible_annotation_gene_sets[collection_name] = {}

    for gene_set_name, genes in gene_sets.items():
        overlapping_genes = genes & consensus_gene_universe_set

        if MIN_GENE_SET_SIZE <= len(overlapping_genes) <= MAX_GENE_SET_SIZE:
            eligible_annotation_gene_sets[collection_name][gene_set_name] = (
                overlapping_genes
            )

In [14]:
# =============================================================================
# Summarize eligible annotation gene sets
# =============================================================================

eligible_annotation_summary = pd.DataFrame(
    [
        {
            "collection": collection_name,
            "analysis_tier": ANNOTATION_COLLECTIONS[collection_name]["analysis_tier"],
            "eligible_gene_set_count": len(gene_sets),
        }
        for collection_name, gene_sets in eligible_annotation_gene_sets.items()
    ]
)

eligible_annotation_summary

,collection,analysis_tier,eligible_gene_set_count
0,HALLMARK,primary,37
1,REACTOME,secondary,272
2,GO_BP,exploratory,2062


## Frozen inferential design

Program annotation uses a competitive rank-based test on the complete signed
consensus-gene profile.

For each consensus program and eligible gene set:

- genes inside the set are compared with all remaining genes in the frozen
  2,389-gene universe;
- the test is two-sided Mann–Whitney U, allowing enrichment toward either the
  positive or negative end of the signed consensus ranking;
- effect size is reported as signed rank-biserial correlation;
- no top-gene threshold is introduced;
- Benjamini–Hochberg correction is applied separately within each
  consensus-program × annotation-collection family;
- Hallmark is the primary annotation layer, Reactome is secondary, and GO
  Biological Process is exploratory.

Statistical enrichment is interpreted as preferential representation within the
frozen transcriptomic program, not as pathway activation, causal regulation, or
biological validation.

In [15]:
# =============================================================================
# Define competitive rank-based enrichment test
# =============================================================================

def test_rank_enrichment(program_weights, gene_set):
    """Test whether a gene set is displaced within a signed program ranking."""
    in_set = program_weights.loc[
        program_weights["gene_symbol"].isin(gene_set),
        "consensus_weight",
    ]
    out_set = program_weights.loc[
        ~program_weights["gene_symbol"].isin(gene_set),
        "consensus_weight",
    ]

    test = mannwhitneyu(
        in_set,
        out_set,
        alternative="two-sided",
    )

    rank_biserial = (
        2 * test.statistic / (len(in_set) * len(out_set))
        - 1
    )

    return test.statistic, test.pvalue, rank_biserial

In [16]:
# =============================================================================
# Compute raw rank-based annotation results
# =============================================================================

annotation_results = []

for program_id, program_weights in ranked_consensus_gene_weights.groupby(
    "consensus_program_id",
    sort=False,
):
    for collection_name, gene_sets in eligible_annotation_gene_sets.items():
        for gene_set_name, gene_set in gene_sets.items():
            statistic, p_value, rank_biserial = test_rank_enrichment(
                program_weights,
                gene_set,
            )

            annotation_results.append(
                {
                    "consensus_program_id": program_id,
                    "collection": collection_name,
                    "analysis_tier": ANNOTATION_COLLECTIONS[
                        collection_name
                    ]["analysis_tier"],
                    "gene_set": gene_set_name,
                    "gene_set_size": len(gene_set),
                    "mann_whitney_u": statistic,
                    "p_value": p_value,
                    "rank_biserial": rank_biserial,
                }
            )

annotation_results = pd.DataFrame(annotation_results)

In [17]:
# =============================================================================
# Apply within-family multiple-testing correction
# =============================================================================

annotation_results["q_value"] = np.nan

for _, family_index in annotation_results.groupby(
    ["consensus_program_id", "collection"]
).groups.items():
    annotation_results.loc[family_index, "q_value"] = multipletests(
        annotation_results.loc[family_index, "p_value"],
        method="fdr_bh",
    )[1]

In [18]:
# =============================================================================
# Summarize annotation results by program and collection
# =============================================================================

annotation_result_summary = (
    annotation_results
    .assign(q_lt_0_05=annotation_results["q_value"].lt(FDR_ALPHA))
    .groupby(
        ["consensus_program_id", "collection", "analysis_tier"],
        as_index=False,
    )
    .agg(
        tested_gene_sets=("gene_set", "size"),
        q_lt_0_05=("q_lt_0_05", "sum"),
        min_rank_biserial=("rank_biserial", "min"),
        max_rank_biserial=("rank_biserial", "max"),
    )
)

annotation_result_summary

,consensus_program_id,collection,analysis_tier,tested_gene_sets,q_lt_0_05,min_rank_biserial,max_rank_biserial
0,CONSENSUS_TX_01,GO_BP,exploratory,2062,465,-0.611515,0.951372
1,CONSENSUS_TX_01,HALLMARK,primary,37,18,-0.512773,0.623029
2,CONSENSUS_TX_01,REACTOME,secondary,272,78,-0.831791,0.937684
3,CONSENSUS_TX_02,GO_BP,exploratory,2062,34,-0.735201,0.534848
4,CONSENSUS_TX_02,HALLMARK,primary,37,9,-0.655918,0.467543
5,CONSENSUS_TX_02,REACTOME,secondary,272,19,-0.732185,0.376198
6,CONSENSUS_TX_03,GO_BP,exploratory,2062,426,-0.570958,0.792200
7,CONSENSUS_TX_03,HALLMARK,primary,37,12,-0.378681,0.745194
8,CONSENSUS_TX_03,REACTOME,secondary,272,53,-0.525102,0.729684


In [19]:
# =============================================================================
# Inspect supported primary Hallmark annotations
# =============================================================================

hallmark_supported_terms = (
    annotation_results.loc[
        (annotation_results["collection"] == "HALLMARK")
        & annotation_results["q_value"].lt(FDR_ALPHA)
    ]
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        ),
        abs_rank_biserial=lambda df: df["rank_biserial"].abs(),
    )
    .sort_values(
        ["consensus_program_id", "abs_rank_biserial"],
        ascending=[True, False],
    )
    [
        [
            "consensus_program_id",
            "gene_set",
            "gene_set_size",
            "direction",
            "rank_biserial",
            "p_value",
            "q_value",
        ]
    ]
)

hallmark_supported_terms

,consensus_program_id,gene_set,gene_set_size,direction,rank_biserial,p_value,q_value
1,CONSENSUS_TX_01,HALLMARK_ALLOGRAFT_REJECTION,53,positive,0.623029,8.053146e-15,1.489832e-13
34,CONSENSUS_TX_01,HALLMARK_UV_RESPONSE_DN,36,negative,-0.512773,1.240909e-07,1.147841e-06
11,CONSENSUS_TX_01,HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION,114,negative,-0.505680,7.251912e-20,2.683207e-18
23,CONSENSUS_TX_01,HALLMARK_INTERFERON_GAMMA_RESPONSE,50,positive,0.465361,1.717907e-08,2.118752e-07
15,CONSENSUS_TX_01,HALLMARK_GLYCOLYSIS,39,negative,-0.383917,3.826992e-05,1.769984e-04
0,CONSENSUS_TX_01,HALLMARK_ADIPOGENESIS,20,negative,-0.375602,3.774420e-03,1.042206e-02
22,CONSENSUS_TX_01,HALLMARK_INTERFERON_ALPHA_RESPONSE,19,positive,0.366644,5.845370e-03,1.346598e-02
28,CONSENSUS_TX_01,HALLMARK_P53_PATHWAY,36,negative,-0.354111,2.609093e-04,1.072627e-03
2,CONSENSUS_TX_01,HALLMARK_ANDROGEN_RESPONSE,19,negative,-0.352787,7.998123e-03,1.644059e-02
18,CONSENSUS_TX_01,HALLMARK_HYPOXIA,58,negative,-0.334798,1.292687e-05,7.166758e-05


In [20]:
# =============================================================================
# Inspect strongest supported secondary Reactome annotations
# =============================================================================

reactome_supported_terms = (
    annotation_results.loc[
        (annotation_results["collection"] == "REACTOME")
        & annotation_results["q_value"].lt(FDR_ALPHA)
    ]
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        ),
        abs_rank_biserial=lambda df: df["rank_biserial"].abs(),
    )
    .sort_values(
        ["consensus_program_id", "abs_rank_biserial"],
        ascending=[True, False],
    )
)

(
    reactome_supported_terms
    .groupby("consensus_program_id", group_keys=False)
    .head(15)
    [
        [
            "consensus_program_id",
            "gene_set",
            "gene_set_size",
            "direction",
            "rank_biserial",
            "q_value",
        ]
    ]
)

,consensus_program_id,gene_set,gene_set_size,direction,rank_biserial,q_value
221,CONSENSUS_TX_01,REACTOME_PHOSPHORYLATION_OF_CD3_AND_TCR_ZETA_C...,14,positive,0.937684,6.289733e-08
139,CONSENSUS_TX_01,REACTOME_GENERATION_OF_SECOND_MESSENGER_MOLECULES,14,positive,0.924090,7.934736e-08
114,CONSENSUS_TX_01,REACTOME_DOWNSTREAM_TCR_SIGNALING,14,positive,0.921564,7.934736e-08
287,CONSENSUS_TX_01,REACTOME_TCR_SIGNALING,18,positive,0.918272,2.457176e-09
54,CONSENSUS_TX_01,REACTOME_ATTACHMENT_OF_BACTERIA_TO_EPITHELIAL_...,11,negative,-0.831791,2.433421e-05
174,CONSENSUS_TX_01,REACTOME_LAMININ_INTERACTIONS,12,negative,-0.813280,1.719269e-05
285,CONSENSUS_TX_01,REACTOME_SYNDECAN_INTERACTIONS,14,negative,-0.729684,3.005542e-05
187,CONSENSUS_TX_01,REACTOME_MET_ACTIVATES_PTK2_SIGNALING,17,negative,-0.676223,2.152064e-05
188,CONSENSUS_TX_01,REACTOME_MET_PROMOTES_CELL_MOTILITY,18,negative,-0.675289,1.300459e-05
205,CONSENSUS_TX_01,REACTOME_NON_INTEGRIN_MEMBRANE_ECM_INTERACTIONS,32,negative,-0.642024,3.798107e-08


In [21]:
# =============================================================================
# Inspect strongest supported exploratory GO-BP annotations
# =============================================================================

go_bp_supported_terms = (
    annotation_results.loc[
        (annotation_results["collection"] == "GO_BP")
        & annotation_results["q_value"].lt(FDR_ALPHA)
    ]
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        ),
        abs_rank_biserial=lambda df: df["rank_biserial"].abs(),
    )
    .sort_values(
        ["consensus_program_id", "abs_rank_biserial"],
        ascending=[True, False],
    )
)

(
    go_bp_supported_terms
    .groupby("consensus_program_id", group_keys=False)
    .head(15)
    [
        [
            "consensus_program_id",
            "gene_set",
            "gene_set_size",
            "direction",
            "rank_biserial",
            "q_value",
        ]
    ]
)

,consensus_program_id,gene_set,gene_set_size,direction,rank_biserial,q_value
1377,CONSENSUS_TX_01,GOBP_PEPTIDE_ANTIGEN_ASSEMBLY_WITH_MHC_CLASS_I...,11,positive,0.951372,1.257660e-06
1378,CONSENSUS_TX_01,GOBP_PEPTIDE_ANTIGEN_ASSEMBLY_WITH_MHC_PROTEIN...,11,positive,0.951372,1.257660e-06
436,CONSENSUS_TX_01,GOBP_B_CELL_RECEPTOR_SIGNALING_PATHWAY,19,positive,0.889185,1.287950e-09
1725,CONSENSUS_TX_01,GOBP_REGULATION_OF_ANTIGEN_RECEPTOR_MEDIATED_S...,19,positive,0.822296,2.776822e-08
369,CONSENSUS_TX_01,GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EX...,16,positive,0.801675,8.160727e-07
370,CONSENSUS_TX_01,GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EX...,16,positive,0.801675,8.160727e-07
371,CONSENSUS_TX_01,GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EX...,16,positive,0.801675,8.160727e-07
374,CONSENSUS_TX_01,GOBP_ANTIGEN_RECEPTOR_MEDIATED_SIGNALING_PATHWAY,45,positive,0.799564,7.370836e-18
435,CONSENSUS_TX_01,GOBP_B_CELL_PROLIFERATION,26,positive,0.792832,2.030318e-10
2060,CONSENSUS_TX_01,GOBP_REGULATION_OF_T_CELL_RECEPTOR_SIGNALING_P...,14,positive,0.774617,1.157170e-05


In [22]:
# =============================================================================
# Consolidate supported biological annotations
# =============================================================================

supported_annotation_results = (
    annotation_results.loc[
        annotation_results["q_value"].lt(FDR_ALPHA)
    ]
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        ),
        abs_rank_biserial=lambda df: df["rank_biserial"].abs(),
    )
    .sort_values(
        [
            "consensus_program_id",
            "analysis_tier",
            "abs_rank_biserial",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

In [23]:
# =============================================================================
# Build compact program-level annotation summary
# =============================================================================

supported_annotation_counts = (
    supported_annotation_results
    .groupby(
        [
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "direction",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "supported_term_count"})
)

program_annotation_summary = (
    supported_annotation_results
    .sort_values("abs_rank_biserial", ascending=False)
    .groupby(
        [
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "direction",
        ],
        as_index=False,
    )
    .first()
    .merge(
        supported_annotation_counts,
        on=[
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "direction",
        ],
        how="left",
    )
    [
        [
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "direction",
            "supported_term_count",
            "gene_set",
            "gene_set_size",
            "rank_biserial",
            "q_value",
        ]
    ]
)

program_annotation_summary

,consensus_program_id,collection,analysis_tier,direction,supported_term_count,gene_set,gene_set_size,rank_biserial,q_value
0,CONSENSUS_TX_01,GO_BP,exploratory,negative,273,GOBP_CARDIAC_EPITHELIAL_TO_MESENCHYMAL_TRANSITION,11,-0.611515,3.844116e-03
1,CONSENSUS_TX_01,GO_BP,exploratory,positive,192,GOBP_PEPTIDE_ANTIGEN_ASSEMBLY_WITH_MHC_CLASS_I...,11,0.951372,1.257660e-06
2,CONSENSUS_TX_01,HALLMARK,primary,negative,13,HALLMARK_UV_RESPONSE_DN,36,-0.512773,1.147841e-06
3,CONSENSUS_TX_01,HALLMARK,primary,positive,5,HALLMARK_ALLOGRAFT_REJECTION,53,0.623029,1.489832e-13
4,CONSENSUS_TX_01,REACTOME,secondary,negative,60,REACTOME_ATTACHMENT_OF_BACTERIA_TO_EPITHELIAL_...,11,-0.831791,2.433421e-05
5,CONSENSUS_TX_01,REACTOME,secondary,positive,18,REACTOME_PHOSPHORYLATION_OF_CD3_AND_TCR_ZETA_C...,14,0.937684,6.289733e-08
6,CONSENSUS_TX_02,GO_BP,exploratory,negative,31,GOBP_KERATINIZATION,24,-0.735201,2.248196e-07
7,CONSENSUS_TX_02,GO_BP,exploratory,positive,3,GOBP_MICROTUBULE_CYTOSKELETON_ORGANIZATION,50,0.288192,3.299579e-02
8,CONSENSUS_TX_02,HALLMARK,primary,negative,6,HALLMARK_INTERFERON_ALPHA_RESPONSE,19,-0.655918,1.512365e-05
9,CONSENSUS_TX_02,HALLMARK,primary,positive,3,HALLMARK_SPERMATOGENESIS,21,0.467543,2.725496e-03


In [24]:
# =============================================================================
# Prepare authoritative program-annotation output
# =============================================================================

program_annotation_output = (
    annotation_results
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        )
    )
    [
        [
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "gene_set",
            "gene_set_size",
            "direction",
            "rank_biserial",
            "mann_whitney_u",
            "p_value",
            "q_value",
        ]
    ]
    .sort_values(
        [
            "consensus_program_id",
            "collection",
            "q_value",
            "gene_set",
        ]
    )
    .reset_index(drop=True)
)

program_annotation_output.shape

(7113, 10)

In [25]:
# =============================================================================
# Program-annotation output path
# =============================================================================

PROGRAM_ANNOTATION_OUTPUT_PATH = (
    OUTPUT_DIR
    / "404_program_annotation_enrichment.csv"
)

In [26]:
# =============================================================================
# Write program-annotation artifact
# =============================================================================

program_annotation_output.to_csv(
    PROGRAM_ANNOTATION_OUTPUT_PATH,
    index=False,
)

In [27]:
# =============================================================================
# Verify written program-annotation artifact
# =============================================================================

reloaded_program_annotation_output = pd.read_csv(
    PROGRAM_ANNOTATION_OUTPUT_PATH
)

reloaded_program_annotation_output.shape

(7113, 10)

## Interpretation and notebook closure

Notebook 404 provides a frozen biological-annotation layer for the three
cross-system consensus transcriptomic representations using prespecified
MSigDB Hallmark, Reactome, and GO Biological Process collections.

The main annotation patterns are:

- `CONSENSUS_TX_01` shows a coherent positive immune-associated axis,
  including antigen presentation, antigen-receptor/T-cell-receptor signaling,
  and interferon-related processes. Its negative extreme contains
  mesenchymal/ECM-associated and other cellular-state programs. This
  immune-associated annotation must be interpreted alongside the lineage
  structure documented upstream; notebook 404 does not establish that this
  axis is lineage-independent or represents a cell-intrinsic immune program.

- `CONSENSUS_TX_02` has a more limited and asymmetric annotation structure.
  Its most reproducible pattern is negative displacement of
  keratinization/epidermal-differentiation and interferon-related processes,
  whereas its positive extreme does not show an equally coherent biological
  theme.

- `CONSENSUS_TX_03` shows a strong positive extracellular-matrix and
  mesenchymal-associated axis, supported across Hallmark, Reactome, and
  GO Biological Process by EMT, collagen organization/metabolism,
  extracellular-matrix assembly, and related structural programs.

These annotations describe preferential representation within the frozen
signed transcriptomic rankings. They do not establish pathway activation,
causal regulation, lineage independence, cell-state origin, or biological
validation.

No consensus program was redefined, reweighted, excluded, rescued, or renamed
based on annotation results.

### Published artifact

- `data/processed/consensus_programs/404_program_annotation_enrichment.csv`

The artifact contains all eligible gene-set tests, including null and
unsupported results, rather than only terms passing the prespecified FDR
threshold. This preserves the complete annotation evidence generated under the
frozen analytical design for downstream interpretation.